# Load Image Datasets Into The Database

# Imports

In [1]:
import sys
from pathlib import Path
import ast
import numpy as np
import pandas as pd

project_root = r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing'
dbtools_root = r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/SqlDatabase'

for path in [project_root, dbtools_root]:
    if path not in sys.path:
        sys.path.append(path)

from dbtools import dbtools as db
from dbtools import load

# Database Connection

In [2]:
try:
    conn = db.connect()
    print("Connected to the database")
except Exception as error:
    print(error)

Connected to the database


# Dataset Type

In [3]:
datasettype_table = db.get_data('datasettypes')

datasettype_table

,id_datasettype,description_datasettype
0,4,2024 Dataset Type. Material mask computed usin...
1,3,2025 Dataset Type. Material mask computed usin...
2,5,Placeholder Dataset Type: A generic category f...
3,6,2025 Dataset Type. Material mask computed usin...
4,7,2026 Dataset Type. Material mask computed usin...


In [4]:
datasettype = 7

# Dataset Folder

In [5]:
folder = Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images')

manifest_paths = sorted(folder.glob('**/image_manifest.csv'))

print(f"Found {len(manifest_paths)} image dataset manifests")
manifest_paths[:5]

Found 59 image dataset manifests


[PosixPath('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.10/registration_139/image_manifest.csv'),
 PosixPath('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.11/registration_140/image_manifest.csv'),
 PosixPath('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.12/registration_141/image_manifest.csv'),
 PosixPath('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.13/registration_142/image_manifest.csv'),
 PosixPath('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.14/registration_143/image_manifest.csv')]

# Already Loaded Registrations

In [6]:
try:
    dataset_registrations_table = db.relation_metadata('datasets', 'registrations', 'dataset_registrations')
    dataset_registrations_table = dataset_registrations_table[
        dataset_registrations_table['datasettype_id_dataset'] == datasettype
    ]
    loaded_registration_ids = set(dataset_registrations_table['id_registration'].astype(int).tolist())
except Exception as e:
    print("No dataset registrations found or error occurred:", e)
    loaded_registration_ids = set()

print(f"Found {len(loaded_registration_ids)} already-loaded registrations for dataset type {datasettype}")

Found 0 already-loaded registrations for dataset type 7


# Helpers

In [7]:
def parse_literal(value, default=None):
    if value is None:
        return default
    if isinstance(value, float) and pd.isna(value):
        return default
    if isinstance(value, (tuple, list)):
        return tuple(value)
    value_str = str(value).strip()
    if not value_str or value_str.lower() == 'nan':
        return default
    try:
        parsed = ast.literal_eval(value_str)
    except Exception:
        return value
    if isinstance(parsed, list):
        return tuple(parsed)
    return parsed


def first_value(df, column, default=None):
    if column not in df.columns:
        return default
    values = df[column].dropna()
    if values.empty:
        return default
    return values.iloc[0]


def image_path_for_kind(df, image_kind):
    rows = df[df['image_kind'] == image_kind]
    if rows.empty:
        return None
    return rows.iloc[0]['image_path']


def tuple_of_ints(value):
    parsed = parse_literal(value)
    if parsed is None:
        return None
    return tuple(int(item) for item in parsed)


def metadata_item(key, value, metadata_type):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    return {
        'key': key,
        'value': str(value),
        'type': metadata_type,
    }


def build_additional_metadata(manifest_df, manifest_path):
    metadata = [
        metadata_item('image_manifest_path', str(manifest_path), 'path'),
        metadata_item('ut_image_path', image_path_for_kind(manifest_df, 'ut'), 'path'),
        metadata_item('volfrac_image_path', image_path_for_kind(manifest_df, 'volfrac'), 'path'),
        metadata_item('areafrac_image_path', image_path_for_kind(manifest_df, 'areafrac'), 'path'),
        metadata_item('sample_id', first_value(manifest_df, 'sample_id'), 'cardinal'),
        metadata_item('sample_name', first_value(manifest_df, 'sample_name'), 'nominal'),
        metadata_item('panel_id', first_value(manifest_df, 'panel_id'), 'cardinal'),
        metadata_item('panel_name', first_value(manifest_df, 'panel_name'), 'nominal'),
        metadata_item('reference_measurement_id', first_value(manifest_df, 'reference_measurement_id'), 'cardinal'),
        metadata_item('registered_measurement_id', first_value(manifest_df, 'registered_measurement_id'), 'cardinal'),
        metadata_item('reference_measurement_path', first_value(manifest_df, 'reference_measurement_path'), 'path'),
        metadata_item('registered_measurement_path', first_value(manifest_df, 'registered_measurement_path'), 'path'),
        metadata_item('ut_image_shape', first_value(manifest_df, 'ut_image_shape'), 'pixels tuple'),
        metadata_item('target_map_shape', first_value(manifest_df, 'target_map_shape'), 'pixels tuple'),
        metadata_item('volfrac_maps_shape', first_value(manifest_df, 'volfrac_maps_shape'), 'pixels tuple'),
        metadata_item('volfrac_region_names', first_value(manifest_df, 'volfrac_region_names'), 'nominal tuple'),
        metadata_item('depth_roi_start_from_frontwall', first_value(manifest_df, 'depth_roi_start_from_frontwall'), 'pixels tuple'),
        metadata_item('depth_roi_end_exclusive_from_frontwall', first_value(manifest_df, 'depth_roi_end_exclusive_from_frontwall'), 'pixels tuple'),
        metadata_item('depth_roi_end_inclusive_from_frontwall', first_value(manifest_df, 'depth_roi_end_inclusive_from_frontwall'), 'pixels tuple'),
        metadata_item('depth_roi_start_z', first_value(manifest_df, 'depth_roi_start_z'), 'pixels tuple'),
        metadata_item('depth_roi_end_exclusive_z', first_value(manifest_df, 'depth_roi_end_exclusive_z'), 'pixels tuple'),
        metadata_item('depth_roi_end_inclusive_z', first_value(manifest_df, 'depth_roi_end_inclusive_z'), 'pixels tuple'),
        metadata_item('material_frontwall_z', first_value(manifest_df, 'material_frontwall_z'), 'pixels'),
        metadata_item('material_backwall_z', first_value(manifest_df, 'material_backwall_z'), 'pixels'),
        metadata_item('material_threshold', first_value(manifest_df, 'material_threshold'), 'ratio'),
        metadata_item('xct_resolution', first_value(manifest_df, 'xct_resolution'), 'mm/pixel'),
        metadata_item('ut_resolution', first_value(manifest_df, 'ut_resolution'), 'mm/pixel'),
    ]
    return [item for item in metadata if item is not None]

# Load Missing Datasets

In [8]:
targets = ['volfrac_front', 'volfrac_middle', 'volfrac_back', 'areafrac']
patch_size = 'no_ut_patches'
description = 'Image-shaped dataset without UT patches, loaded from existing image manifests.'

results = []

for manifest_path in manifest_paths:
    manifest_df = pd.read_csv(manifest_path)

    if manifest_df.empty:
        results.append({
            'manifest_path': str(manifest_path),
            'status': 'empty_manifest',
            'dataset_id': None,
            'registration_id': None,
        })
        continue

    registration_id = int(first_value(manifest_df, 'registration_id'))

    if registration_id in loaded_registration_ids:
        print(f"Registration {registration_id} is already loaded for dataset type {datasettype}, skipping")
        results.append({
            'manifest_path': str(manifest_path),
            'status': 'already_loaded',
            'dataset_id': None,
            'registration_id': registration_id,
        })
        continue

    reconstruction_shape = tuple_of_ints(first_value(manifest_df, 'volfrac_maps_shape'))
    if reconstruction_shape is None:
        volfrac_path = image_path_for_kind(manifest_df, 'volfrac')
        raise ValueError(f"Missing volfrac_maps_shape in {manifest_path}; volfrac path: {volfrac_path}")

    rows = int(np.prod(reconstruction_shape))
    additional_metadata = build_additional_metadata(manifest_df, manifest_path)

    print(f"Loading registration {registration_id} from {manifest_path}")
    dataset_id = load.load_dataset(
        conn,
        datasettype_id=int(datasettype),
        file_path=str(manifest_path),
        rows=rows,
        patch_size=patch_size,
        targets=targets,
        reconstruction_shape=reconstruction_shape,
        registration_ids=[registration_id],
        description=description,
        additional_metadata=additional_metadata,
    )

    status = 'loaded' if dataset_id != -1 else 'error'
    if status == 'loaded':
        loaded_registration_ids.add(registration_id)

    results.append({
        'manifest_path': str(manifest_path),
        'status': status,
        'dataset_id': dataset_id,
        'registration_id': registration_id,
        'rows': rows,
        'reconstruction_shape': reconstruction_shape,
    })

results_df = pd.DataFrame(results)
results_df

Loading registration 139 from /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.10/registration_139/image_manifest.csv
Dataset from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.10/registration_139/image_manifest.csv' loaded with ID: 1098
Loading registration 140 from /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.11/registration_140/image_manifest.csv
Dataset from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.11/registration_140/image_manifest.csv' loaded with ID: 1099
Loading registration 141 from /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.12/registration_141/image_manifest.csv
Dataset from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images/1.12/registration_141/image_manifest.csv' loaded with ID: 1100
Load

,manifest_path,status,dataset_id,registration_id,rows,reconstruction_shape
0,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1098,139,9360,"(3, 80, 39)"
1,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1099,140,9600,"(3, 80, 40)"
2,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1100,141,9000,"(3, 75, 40)"
3,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1101,142,10062,"(3, 78, 43)"
4,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1102,143,9840,"(3, 80, 41)"
5,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1103,144,9480,"(3, 79, 40)"
6,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1104,145,9243,"(3, 79, 39)"
7,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1105,146,9120,"(3, 80, 38)"
8,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1106,147,9000,"(3, 75, 40)"
9,/home/alberto.vicente/phd/DataDriven_UT_Albert...,loaded,1107,148,9450,"(3, 75, 42)"


# Summary

In [9]:
results_df['status'].value_counts(dropna=False)

status
loaded    59
Name: count, dtype: int64